# GA4 Ecommerce Funnel Data Prep


Evelina Ramoskaite

### Extraction

In [2]:
# Import libraries
from google.cloud import bigquery # reference readme file for setup/authentication instructions
import pandas as pd


In [3]:
# Specifying Project ID 
PROJECT_ID = 'bigquery-public-evelinar'
client = bigquery.Client(project=PROJECT_ID)

In [4]:
# Set Start and End Date
# Format: YYYYMMDD
START_DATE = '20210101'
END_DATE = '20210131'


In [29]:
# Extracting data from Bigquery 
with open('checkout_funnel_daily.sql') as f:
     query = f.read().format(start_date=START_DATE, end_date=END_DATE)
df = client.query(query).to_dataframe()


In [30]:
print(df.shape)
df.head(10)

(238073, 12)


,event_date,event_name,user_pseudo_id,session_id,event_ts,country,state,device_category,traffic_medium,traffic_source,transaction_id,revenue
0,20210102,session_start,1005484.1092567297,2718913892,2021-01-02 06:16:22.969088+00:00,United States,Massachusetts,desktop,organic,google,NaN,NaN
1,20210102,session_start,1019468.5334749980,2306134442,2021-01-02 12:09:21.129199+00:00,United States,Washington,mobile,(data deleted),(data deleted),NaN,NaN
2,20210102,session_start,1019468.5334749980,7900311379,2021-01-02 11:53:54.999615+00:00,United States,Washington,mobile,<Other>,<Other>,NaN,NaN
3,20210102,session_start,1020695.6921096883,1906836157,2021-01-02 10:16:01.969864+00:00,United States,Washington,desktop,organic,google,NaN,NaN
4,20210102,session_start,1034552.3956022963,7718736252,2021-01-02 15:06:48.394960+00:00,Brazil,State of Minas Gerais,desktop,organic,google,NaN,NaN
5,20210102,session_start,1035433.9962487028,7548993251,2021-01-02 19:32:18.390610+00:00,United States,Texas,mobile,organic,google,NaN,NaN
6,20210102,session_start,1054184.5156674867,2757256400,2021-01-02 20:29:52.891496+00:00,Ireland,County Dublin,desktop,<Other>,<Other>,NaN,NaN
7,20210102,add_payment_info,1055969.5872512303,9503003312,2021-01-02 02:12:15.013124+00:00,United States,Florida,mobile,(none),(direct),(not set),NaN
8,20210102,session_start,1055969.5872512303,5298604884,2021-01-02 20:14:17.635123+00:00,United States,Florida,mobile,referral,shop.googlemerchandisestore.com,NaN,NaN
9,20210102,session_start,1055969.5872512303,9503003312,2021-01-02 02:09:31.057472+00:00,United States,Florida,mobile,(none),(direct),NaN,NaN


## Cleaning

In [31]:
# Format Date
df['event_date'] = pd.to_datetime(df['event_date'], format = '%Y%m%d')

In [32]:
# creating a unique session key
# session_id by itself is just time-based
df['session_key'] = df['user_pseudo_id'] + '-' + df['session_id'].astype(str)
df = df.drop(columns=['user_pseudo_id', 'session_id'])

In [33]:
# Dropping Duplicate events, if any
print(len(df))
df = df.drop_duplicates(subset= ['session_key','event_name','event_ts'])
print(len(df))

238073
238073


In [34]:
#Checking Missing Values
df.isna().sum()

event_date              0
event_name              0
event_ts                0
country                 0
state                   0
device_category         0
traffic_medium          0
traffic_source          0
transaction_id     116549
revenue            237169
session_key             0
dtype: int64

In [35]:
# 2. Get unique values 
for c in ['traffic_source','traffic_medium','device_category']:
    print(c)
    print(df[c].unique().tolist())

traffic_source
['google', '(data deleted)', '<Other>', '(direct)', 'shop.googlemerchandisestore.com']
traffic_medium
['organic', '(data deleted)', '<Other>', '(none)', 'referral', 'cpc']
device_category
['desktop', 'mobile', 'tablet']


In [36]:
# defining channel names
def channel(row):
    src = row['traffic_source']
    med = row['traffic_medium']

    if src == 'shop.googlemerchandisestore.com':
        return 'Unattributed'
    if med == 'organic':
        return 'Organic Search'
    if med == 'cpc':
        return 'Paid Search'
    if med == 'referral':
        return 'Referral'
    if src == '(direct)':
        return 'Direct'
    if '(data deleted)' in (src, med):
        return 'Unattributed'
    return 'Unattributed'

df['channel'] = df.apply(channel, axis=1)

### Funnel Data Prep

In [37]:
FUNNEL_STEPS = [
    'session_start',
    'view_item',
    'add_to_cart',
    'begin_checkout',
    'add_shipping_info',
    'add_payment_info',
    'purchase',
]
LABELS = {
    'session_start': 'Session',
    'view_item': 'Product View',
    'add_to_cart': 'Add to Cart',
    'begin_checkout': 'Checkout Started',
    'add_shipping_info': 'Shipping Info',
    'add_payment_info': 'Payment Info',
    'purchase': 'Purchase',
}

STEP_ORDER = {
    'session_start': 1,
    'view_item': 2,
    'add_to_cart': 3,
    'begin_checkout': 4,
    'add_shipping_info': 5,
    'add_payment_info': 6,
    'purchase': 7,
}

In [38]:
df['step'] = df['event_name'].map(LABELS)
df['step_order'] = df['event_name'].map(STEP_ORDER)

In [39]:
# Aggregating funnel data. 
funnel = df.groupby(['event_date','device_category','step','step_order'])['session_key'].nunique().reset_index(name = 'sessions')

In [40]:
#Check
funnel.groupby('step_order')['sessions'].sum()

step_order
1    116514
2     23153
3      4548
4      2161
5      2161
6      1563
7      1116
Name: sessions, dtype: int64

### Purchase Data Prep

In [41]:
# Aggregating purchase data
purchases = df[df['event_name'] == 'purchase'].drop_duplicates(subset= ['transaction_id','session_key'])
pur_df = purchases.groupby(['event_date','country','device_category','channel'])['revenue'].sum().reset_index()

In [42]:
purchases.head(5)

,event_date,event_name,event_ts,country,state,device_category,traffic_medium,traffic_source,transaction_id,revenue,session_key,channel,step,step_order
3211,2021-01-02,purchase,2021-01-02 20:17:19.079970+00:00,United States,Florida,mobile,referral,shop.googlemerchandisestore.com,792636,7.0,1055969.5872512303-5298604884,Unattributed,Purchase,7
3233,2021-01-02,purchase,2021-01-02 03:25:06.030951+00:00,United States,Colorado,mobile,organic,google,562964,13.0,3024189.6101670146-7859325572,Organic Search,Purchase,7
3253,2021-01-02,purchase,2021-01-02 19:18:34.166890+00:00,United States,Pennsylvania,mobile,organic,google,(not set),NaN,7296845.4997447007-7961433211,Organic Search,Purchase,7
3279,2021-01-02,purchase,2021-01-02 18:04:51.634113+00:00,Canada,Manitoba,mobile,<Other>,<Other>,704467,48.0,42807215.0722008548-7913682591,Unattributed,Purchase,7
3285,2021-01-02,purchase,2021-01-02 16:28:58.657567+00:00,United States,Massachusetts,tablet,organic,google,595162,42.0,43382307.1290802082-7744018777,Organic Search,Purchase,7


### Export

In [43]:
# Save to CSV
funnel.to_csv('Funnel.csv', index=False)
pur_df.to_csv('Purchases.csv',index=False)